# 1.0 POI dataset exploration

Structure, opening hours, family flags and durations of the 50 curated Corfu POIs, plus the synthetic rain fixture.
Opening hours, durations and family flags are sample data, not live visitor information.

In [1]:
import json

from datetime import date
from collections import Counter

import pandas as pd

from corga.paths import DATASET_PATH, RAIN_FIXTURE_PATH
from corga.data.hours import WEEKDAYS
from corga.data.store import POIStore

raw = json.loads(DATASET_PATH.read_text(encoding="utf-8"))
store = POIStore.from_json(DATASET_PATH)
pois = store.all()
print(f"{raw['dataset']} v{raw['version']}: {len(pois)} POIs, timezone {raw['timezone']}")

Corfu points of interest v2.0: 50 POIs, timezone Europe/Athens


## 1. Records and fields

In [2]:
df = pd.DataFrame(
    {
        "id": p.id,
        "name": p.name,
        "category": p.category,
        "setting": p.setting,
        "minutes": p.suggested_visit_minutes,
        "family": p.family_suitable,
        "hours_known": p.opening_hours is not None,
        "tags": ", ".join(p.tags),
    }
    for p in pois
).set_index("id")
df.head(10)

,name,category,setting,minutes,family,hours_known,tags
id,,,,,,,
CF-001,Old Fortress,history,mixed,85,None,True,"fortress, venetian, viewpoint"
CF-002,New Fortress,history,outdoor,65,None,True,"fortress, venetian, viewpoint"
CF-003,Kremasti Square and Cistern,history,outdoor,20,True,True,"architecture, venetian, square"
CF-004,Saint Spyridon Church,history,indoor,25,None,False,"religion, art, architecture"
CF-005,Campiello Quarter,history,outdoor,35,True,True,"walking, architecture, venetian"
CF-006,Maitland Rotunda,history,outdoor,15,True,True,"architecture, monument, square"
CF-007,Casa Parlante Museum,history,indoor,55,True,True,"museum, family, history"
CF-008,Gate of the New Fortress,history,outdoor,15,None,True,"fortress, venetian, architecture"
CF-009,Old Town Hall (Loggia dei Nobili),history,outdoor,20,True,True,"architecture, theatre, history"


In [3]:
for field, note in raw["fields_note"].items():
    print(f"- {field}: {note}\n")

- opening_hours: Weekday schedule mon-sun in local time. Null whole field = no confirmed opening window. Null weekday = closed. End earlier than start (e.g. 20:00-02:00) means next calendar day. 00:00-24:00 = continuous access.

- setting: indoor, outdoor or mixed; coarse exposure class for weather-aware planning. Mixed is not guaranteed indoor shelter.

- suggested_visit_minutes: Estimated visit or meal duration, not an advertised visitor duration.

- family_suitable: true = suitable for families in principle; false = left out of plans for a child; null = unspecified. Planning flags, not real-world accessibility or safety certifications.

- source_urls: Research references for evergreen background details. Not a statement that a shop, museum, event or outdoor space is currently accessible.

- area: Approximate locality label from editorial review; use coordinates for proximity checks, not area equality.

- description_provenance: Editorial trace of what verifies the description. A pla

## 2. Categories, settings and tags

There is no `art` or `museum` category; those appear only as tags.

In [4]:
pd.concat(
    [df["category"].value_counts().rename("category"), df["setting"].value_counts().rename("setting")], axis=1
).fillna("")

,category,setting
history,24.0,
nature,10.0,
food,9.0,
local_market,4.0,
nightlife,3.0,
outdoor,,22.0
indoor,,21.0
mixed,,7.0


In [5]:
tag_counts = Counter(t for p in pois for t in p.tags)
pd.Series(tag_counts).sort_values(ascending=False).head(15).rename("POIs").to_frame()

,POIs
architecture,12
venetian,8
walking,8
museum,8
restaurant,7
viewpoint,5
fortress,5
art,5
outside_centre,5
religion,4


In [6]:
for tag in ["museum", "art", "gallery", "painting", "venetian"]:
    print(f"{tag:>9}: {[p.id for p in pois if tag in p.tags]}")

   museum: ['CF-007', 'CF-012', 'CF-014', 'CF-017', 'CF-021', 'CF-022', 'CF-029', 'CF-031']
      art: ['CF-004', 'CF-012', 'CF-014', 'CF-017', 'CF-045']
  gallery: ['CF-014']
 painting: ['CF-045']
 venetian: ['CF-001', 'CF-002', 'CF-003', 'CF-005', 'CF-008', 'CF-026', 'CF-027', 'CF-028']


## 3. Opening hours

Whole-field `null` → unknown; `null` weekday → closed; end < start → overnight; `00:00-24:00` → continuous.

In [7]:
unknown = df[~df["hours_known"]][["name", "category", "setting"]]
print(f"{len(unknown)} POIs without confirmed hours")
unknown

9 POIs without confirmed hours


,name,category,setting
id,,,
CF-004,Saint Spyridon Church,history,indoor
CF-013,Historic Jewish Synagogue,history,indoor
CF-018,Patounis' Soap Factory,local_market,indoor
CF-026,Church of St. John the Baptist in Sternes,history,indoor
CF-037,Mandraki Marina,nature,outdoor
CF-045,Lotos / Alcia Art Space,local_market,indoor
CF-046,Ilios Jewellery Workshop,local_market,indoor
CF-047,Perdita's Glass Art,local_market,indoor
CF-050,Municipal and Regional Theatre of Corfu,nightlife,indoor


In [8]:
def kind(p):
    if p.opening_hours is None:
        return "unknown"
    specs = {s for s in p.opening_hours.root.values() if s}
    if specs == {"00:00-24:00"}:
        return "continuous"
    if any(s.split("-")[1] < s.split("-")[0] for s in specs):
        return "overnight"
    return "daytime"


df["hours_kind"] = [kind(p) for p in pois]
df["hours_kind"].value_counts().to_frame()

,count
hours_kind,
continuous,20
daytime,19
unknown,9
overnight,2


Weekday hours of the POIs with a daytime or overnight schedule (`closed` = null weekday).

In [9]:
timed = [p for p in pois if df.loc[p.id, "hours_kind"] in ("daytime", "overnight")]
matrix = pd.DataFrame(
    {day: [p.opening_hours[day] or "closed" for p in timed] for day in WEEKDAYS},
    index=[f"{p.id} {p.name[:28]}" for p in timed],
)
matrix

,mon,tue,wed,thu,fri,sat,sun
CF-001 Old Fortress,09:00-17:00,09:00-17:00,09:00-17:00,09:00-17:00,09:00-17:00,09:00-17:00,09:00-17:00
CF-002 New Fortress,09:00-15:30,closed,09:00-15:30,09:00-15:30,09:00-15:30,09:00-15:30,09:00-15:30
CF-007 Casa Parlante Museum,10:00-17:00,10:00-17:00,10:00-17:00,10:00-17:00,10:00-17:00,10:00-17:00,10:00-17:00
CF-012 Corfu Philharmonic Music Mus,10:00-16:00,closed,10:00-16:00,10:00-16:00,10:00-16:00,10:00-16:00,10:00-16:00
CF-014 Municipal Art Gallery of Cor,closed,10:00-16:00,10:00-16:00,10:00-16:00,10:00-16:00,10:00-16:00,10:00-16:00
CF-017 Museum of Asian Art,closed,10:00-17:00,10:00-17:00,10:00-17:00,10:00-17:00,10:00-17:00,10:00-17:00
CF-021 Achilleion Palace,09:00-16:00,09:00-16:00,09:00-16:00,09:00-16:00,09:00-16:00,09:00-16:00,09:00-16:00
CF-022 Mon Repos Museum of Palaiopo,09:00-16:00,09:00-16:00,09:00-16:00,09:00-16:00,09:00-16:00,09:00-16:00,09:00-16:00
CF-024 Boiler Food Bar,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00
CF-025 Greek Market Food,11:30-19:00,11:30-19:00,11:30-19:00,11:30-19:00,11:30-19:00,11:30-19:00,11:30-19:00


In [10]:
days = {
    "Mon 5 Oct": date(2026, 10, 5),
    "Tue 6 Oct": date(2026, 10, 6),
    "Wed 7 Oct": date(2026, 10, 7),
}
for label, day in days.items():
    closed = [f"{p.id} {p.name}" for p in timed if not p.opening_hours.is_open_on(day)]
    print(f"{label}: closed → {closed or 'none'}")

Mon 5 Oct: closed → ['CF-014 Municipal Art Gallery of Corfu', 'CF-017 Museum of Asian Art', 'CF-031 Archaeological Museum of Corfu']
Tue 6 Oct: closed → ['CF-002 New Fortress', 'CF-012 Corfu Philharmonic Music Museum', 'CF-029 Byzantine Museum (Antivouniotissa)']
Wed 7 Oct: closed → none


## 4. Family flag and visit durations

In [11]:
df["family"].value_counts(dropna=False).rename_axis("family_suitable").to_frame("POIs")

,POIs
family_suitable,
True,27
None,21
False,2


In [12]:
df["minutes"].describe().to_frame("suggested_visit_minutes").T

,count,mean,std,min,25%,50%,75%,max
suggested_visit_minutes,50.0,40.4,19.452716,15.0,25.0,35.0,55.0,90.0


Visit durations of five named places in a 90-minute window:

In [13]:
named_ids = ["CF-001", "CF-002", "CF-021", "CF-022", "CF-017"]
minutes = {i: store.get(i).suggested_visit_minutes for i in named_ids}
print(minutes, "→ total", sum(minutes.values()), "min vs 90")

{'CF-001': 85, 'CF-002': 65, 'CF-021': 80, 'CF-022': 60, 'CF-017': 65} → total 355 min vs 90


## 5. Rain fixture

Hourly conditions for Wed 7 Oct 2026 (Europe/Athens local time).

In [14]:
rain = json.loads(RAIN_FIXTURE_PATH.read_text(encoding="utf-8"))
print(rain["scenario"])
pd.DataFrame(rain["hourly"]).set_index("local_time")

Rain begins after 15:00 local time; use for weather-driven itinerary re-planning tests.


,temperature_c,precipitation_mm,weather_code
local_time,,,
2026-10-07T13:00:00,23,0.0,1
2026-10-07T14:00:00,23,0.0,2
2026-10-07T15:00:00,21,3.2,63
2026-10-07T16:00:00,20,4.1,63
2026-10-07T17:00:00,20,1.8,61
